# FedUCA Phase 1b: ViT Embedding Cache

**Before running:** Add dataset `kmader/skin-cancer-mnist-ham10000` | GPU T4 | Internet ON

In [ ]:
import os, json, time
import numpy as np
import pandas as pd
import torch
import torchvision.transforms as T
from torchvision.models import vit_b_16, ViT_B_16_Weights
from torch.utils.data import DataLoader, Dataset
from PIL import Image
from collections import Counter
from sklearn.model_selection import GroupShuffleSplit

DEVICE   = "cuda" if torch.cuda.is_available() else "cpu"
SAVE_DIR = "/kaggle/working/ham10000_vit_embeddings"
os.makedirs(SAVE_DIR, exist_ok=True)
CLASS_NAMES = ['akiec','bcc','bkl','df','mel','nv','vasc']
print(f"Device: {DEVICE}")

In [ ]:
# Load frozen ViT-B/16 SWAG backbone
print("Loading ViT-B/16 SWAG weights...")
weights  = ViT_B_16_Weights.IMAGENET1K_SWAG_E2E_V1
backbone = vit_b_16(weights=weights).to(DEVICE)
backbone.eval()
for p in backbone.parameters():
    p.requires_grad = False
print(f"Frozen. Params: {sum(p.numel() for p in backbone.parameters()):,}")
transform = T.Compose([
    T.Resize(384, interpolation=T.InterpolationMode.BICUBIC),
    T.CenterCrop(384),
    T.ToTensor(),
    T.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),
])
print("Transform ready.")

In [ ]:
# ── DIAGNOSTIC: show everything under /kaggle/input ─────────────────
print("Contents of /kaggle/input:")
for item in sorted(os.listdir("/kaggle/input")):
    full = os.path.join("/kaggle/input", item)
    print(f"  {item}/")
    if os.path.isdir(full):
        sub = os.listdir(full)
        for s in sorted(sub)[:20]:
            sfull = os.path.join(full, s)
            if os.path.isdir(sfull):
                n = len(os.listdir(sfull))
                print(f"    {s}/  ({n} files)")
            else:
                print(f"    {s}  ({os.path.getsize(sfull)//1024} KB)")

In [ ]:
# ── Auto-find metadata CSV and images anywhere under /kaggle/input ──
import glob

# Search for the metadata CSV
csv_hits = glob.glob("/kaggle/input/**/*metadata*.csv", recursive=True)
csv_hits += glob.glob("/kaggle/input/**/*HAM*.csv",     recursive=True)
print("CSV files found:", csv_hits)

# Search for jpg images
jpg_sample = glob.glob("/kaggle/input/**/*.jpg", recursive=True)[:5]
print(f"Sample jpg paths ({len(glob.glob('/kaggle/input/**/*.jpg',recursive=True))} total):")
for p in jpg_sample:
    print(" ", p)

In [ ]:
# ── Load metadata from whichever path was found ──────────────────────
if not csv_hits:
    raise FileNotFoundError(
        "No metadata CSV found under /kaggle/input. "
        "Make sure you added the kmader/skin-cancer-mnist-ham10000 dataset "
        "BEFORE running: go to the notebook sidebar, click Add Data, "
        "search 'skin-cancer-mnist-ham10000', click Add.")

meta_path = csv_hits[0]
print(f"Using metadata: {meta_path}")
meta = pd.read_csv(meta_path)
meta['label'] = meta['dx'].map({c:i for i,c in enumerate(CLASS_NAMES)})
print(f"Total images : {len(meta)}")
print(f"Unique lesions: {meta['lesion_id'].nunique()}")
print(meta['dx'].value_counts().to_string())

# Index all jpg images
all_jpgs = glob.glob("/kaggle/input/**/*.jpg", recursive=True)
img_lookup = {os.path.splitext(os.path.basename(p))[0]: p for p in all_jpgs}
print(f"\nImages indexed: {len(img_lookup)}")
missing = [r.image_id for _,r in meta.iterrows() if r.image_id not in img_lookup]
print(f"Missing       : {len(missing)}")

In [ ]:
# Lesion-disjoint split
gss1 = GroupShuffleSplit(n_splits=1, test_size=0.10, random_state=42)
tv_idx, te_idx = next(gss1.split(meta, meta['label'], groups=meta['lesion_id']))
meta_tv, meta_test = meta.iloc[tv_idx].reset_index(drop=True), meta.iloc[te_idx].reset_index(drop=True)

gss2 = GroupShuffleSplit(n_splits=1, test_size=0.111, random_state=42)
tr_idx, va_idx = next(gss2.split(meta_tv, meta_tv['label'], groups=meta_tv['lesion_id']))
meta_train = meta_tv.iloc[tr_idx].reset_index(drop=True)
meta_val   = meta_tv.iloc[va_idx].reset_index(drop=True)

assert len(set(meta_train.lesion_id) & set(meta_val.lesion_id))  == 0
assert len(set(meta_train.lesion_id) & set(meta_test.lesion_id)) == 0
assert len(set(meta_val.lesion_id)   & set(meta_test.lesion_id)) == 0
print("ZERO lesion overlap confirmed")
for name,df in [("Train",meta_train),("Val",meta_val),("Test",meta_test)]:
    print(f"  {name}: {len(df)} images | {df.lesion_id.nunique()} lesions")

pd.concat([meta_train.assign(split='train'),
           meta_val.assign(split='val'),
           meta_test.assign(split='test')]
).to_csv(os.path.join(SAVE_DIR,"lesion_split_index.csv"), index=False)
print("lesion_split_index.csv saved")

In [ ]:
# Dataset + extraction
class HAMDataset(Dataset):
    def __init__(self, df, lookup, tfm):
        self.df=df.reset_index(drop=True); self.lookup=lookup; self.tfm=tfm
    def __len__(self): return len(self.df)
    def __getitem__(self, i):
        r = self.df.iloc[i]
        p = self.lookup.get(r.image_id)
        img = Image.open(p).convert('RGB') if p else Image.new('RGB',(384,384))
        return self.tfm(img), int(r.label), str(r.lesion_id), str(r.image_id)

@torch.no_grad()
def extract(df, name, bs=64):
    ds = HAMDataset(df, img_lookup, transform)
    dl = DataLoader(ds, batch_size=bs, shuffle=False, num_workers=2, pin_memory=True)
    embs,lbls,les,ids = [],[],[],[]
    t0 = time.time()
    for bi,(imgs,labels,lesions,imgids) in enumerate(dl):
        x = backbone._process_input(imgs.to(DEVICE))
        n = x.shape[0]
        x = torch.cat([backbone.class_token.expand(n,-1,-1), x], dim=1)
        cls = backbone.encoder(x)[:,0,:].cpu().float().numpy()
        embs.append(cls); lbls.append(labels.numpy())
        les.extend(lesions); ids.extend(imgids)
        done = min((bi+1)*bs, len(ds))
        if (bi+1) % 20 == 0:
            ela = time.time()-t0
            print(f"  {name}: {done}/{len(ds)} | {ela:.0f}s | ETA {ela/done*(len(ds)-done):.0f}s")
    e=np.concatenate(embs); l=np.concatenate(lbls)
    print(f"  DONE {name}: {len(e)} embeddings | {time.time()-t0:.0f}s")
    return e,l,les,ids

print("Ready.")

In [ ]:
# Extract all splits
print("="*55)
print("EXTRACTING EMBEDDINGS")
print("="*55)
split_meta={}
for name,df in [("train",meta_train),("val",meta_val),("test",meta_test)]:
    print(f"\n--- {name} ({len(df)} images) ---")
    e,l,les,ids = extract(df,name)
    dist = {CLASS_NAMES[k]:int(v) for k,v in Counter(l.tolist()).items()}
    print(f"  Classes: {dist}")
    out = os.path.join(SAVE_DIR,f"embeddings_{name}.npz")
    np.savez_compressed(out, embeddings=e, labels=l,
                        lesion_ids=np.array(les,dtype=object),
                        image_ids =np.array(ids,dtype=object))
    mb = os.path.getsize(out)/1e6
    print(f"  Saved {out} ({mb:.1f} MB)")
    split_meta[name]={"n_samples":int(len(e)),"n_lesions":int(len(set(les))),
                      "class_counts":dist,"size_mb":round(mb,2)}

In [ ]:
# Metadata + sanity check
with open(os.path.join(SAVE_DIR,"metadata.json"),"w") as f:
    json.dump({"model":"ViT-B/16 SWAG E2E","input_size":"384x384 bicubic",
               "embed_dim":768,"dataset":"kmader/skin-cancer-mnist-ham10000",
               "split_protocol":"lesion-disjoint GroupShuffleSplit seed=42",
               "num_classes":7,"class_names":CLASS_NAMES,"splits":split_meta},f,indent=2)

chk=np.load(os.path.join(SAVE_DIR,"embeddings_train.npz"),allow_pickle=True)
e,l=chk['embeddings'],chk['labels']
norms=np.linalg.norm(e,axis=1)
mel=np.where(l==4)[0]; nv=np.where(l==5)[0]
cos=lambda a,b: float(np.dot(a,b)/(np.linalg.norm(a)*np.linalg.norm(b)+1e-9))
smm=np.mean([cos(e[mel[i]],e[mel[i+1]]) for i in range(10)])
smn=np.mean([cos(e[mel[i]],e[nv[i]])    for i in range(10)])

print("="*55); print("SANITY CHECK"); print("="*55)
print(f"Train shape : {e.shape}")
print(f"L2 norm     : mean={norms.mean():.2f}  std={norms.std():.2f}")
print(f"mel-mel sim : {smm:.4f}")
print(f"mel-nv  sim : {smn:.4f}  {'OK' if smm>smn else 'WARNING'}")
print()
print("FILES:")
for fname in sorted(os.listdir(SAVE_DIR)):
    print(f"  {fname}  ({os.path.getsize(os.path.join(SAVE_DIR,fname))/1e6:.1f} MB)")
print()
for k,v in split_meta.items():
    print(f"  {k}: {v['n_samples']} images | {v['n_lesions']} lesions")
print()
print("DONE. Output tab -> New Dataset -> ham10000-vit-embeddings-original")